In [4]:
import os
import sys
import torch

# Jupyter kernels cache modules in sys.modules.
# To prevent conflicts between model/config modules from different folders (e.g., deepseek3 vs qwen3_5),
# we remove any cached modules sharing these names before importing the new one.
for mod in ["model", "config", "block", "attention", "mlp", "rotary", "rms_norm", "tokenizer"]:
    sys.modules.pop(mod, None)

# Add model architecture directory to path
sys.path.insert(0, "qwen3_5")
from model import TinyQwen35
from config import ModelConfig

# Add root directory to path to import the new BPE tokenizer
sys.path.insert(0, ".")
from train_tokenizer import BPETokenizer

In [6]:
# Load the newly trained BPE tokenizer
tokenizer_path = os.path.join("data", "bpe_tokenizer.json")
tok = BPETokenizer(tokenizer_path)
print(f"Loaded BPETokenizer. Vocab size: {tok.vocab_size}")

# Configure the Qwen3.5 model using the BPE vocabulary size
cfg = ModelConfig(vocab_size=tok.vocab_size)
model = TinyQwen35(cfg)

# Load checkpoint if it exists, otherwise initialize randomly
checkpoint_path = os.path.join("qwen3_5", "tiny_qwen35.pt")
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    print(f"Successfully loaded trained checkpoint from {checkpoint_path}")
else:
    print("No trained checkpoint found. Using randomly initialized model.")

model.eval()
model

Loaded BPETokenizer. Vocab size: 2000
Successfully loaded trained checkpoint from qwen3_5/tiny_qwen35.pt


TinyQwen35(
  (embed_tokens): Embedding(2000, 32)
  (layers): ModuleList(
    (0-2): 3 x TransformerBlock(
      (input_layernorm): RMSNorm()
      (mixer): GatedDeltaNet(
        (q_proj): Linear(in_features=32, out_features=32, bias=False)
        (k_proj): Linear(in_features=32, out_features=32, bias=False)
        (v_proj): Linear(in_features=32, out_features=32, bias=False)
        (o_proj): Linear(in_features=32, out_features=32, bias=False)
        (alpha_proj): Linear(in_features=32, out_features=4, bias=True)
        (beta_proj): Linear(in_features=32, out_features=4, bias=True)
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=32, out_features=64, bias=False)
        (up_proj): Linear(in_features=32, out_features=64, bias=False)
        (down_proj): Linear(in_features=64, out_features=32, bias=False)
      )
    )
    (3): TransformerBlock(
      (input_layernorm): RMSNorm()
      (mixer): Attention(
        (q_proj)

In [7]:
# Generate movie titles (each sequence starts with the <bos> token)
start = torch.full((10, 1), tok.bos_token_id, dtype=torch.long)
out = model.generate(start, max_new_tokens=15, 
                     temperature=0.8, eos_id=tok.eos_token_id)
out

tensor([[   2, 1357,  129, 1782, 1597,  145,   29,  123,  618,  246,   29,   50,
          118, 1305, 1739, 1320],
        [   2,  347, 1091,  109,  516,  109,  743,  222,   53,  172, 1601,  113,
          264,  552,  109,  180],
        [   2,  109,  781,  109,  492,  275,  109,  132,  214,  186,  435,  109,
          686,  119,  109, 1733],
        [   2,  109,  913,  109,   56,  109,  106,  119,  244,  409,  131,   38,
          454,  332,  119,  109],
        [   2, 1057,  119,  109,  919,  104,  192, 1245,  228,   17,   29,  161,
          162,  109,   65,  594],
        [   2,  109,  126,  614,   47,  109,  302,  651,  109,  695,  526,  119,
          109,  164,  119,  109],
        [   2,  131, 1086,  132,  145,  372,  132,   50,  109,  595,   48,  371,
           29,  153,  377,  316],
        [   2, 1281,  150,   46,   41, 1079,  656, 1341, 1346,  878,  315,  159,
          172, 1718,   29,  310],
        [   2,  129,  240,   29,  389,  175,  109,  961,  166,  301,  119,  109,

In [8]:
# Decode and print the generated movie titles
for row in out.tolist():
    # Remove the starting <bos> token ID when decoding
    decoded_title = tok.decode(row[1:])
    print(decoded_title)

 kiss a resur load: d bri se:o brol ben mach
 anhood the god the feverr krifen ro island the v
 theght the comos the hart chash theio of the mut
 the sea theu thein of forakeescadety of the
 gl of the heartthe j pre bl.:'s and the� death
 the fjal thequann the first back of these of the
escre hadte ho the yourm story: n al world
waysigkf american pieough los earth 1ch kuff: with
 a man: exent the rebet day of the godicec wall
 the host the fav iellg vampire the tran10 doraemon
